In [ ]:
#!pip install nest_asyncio
#!pip install pyppeteer
#!pip install -U langgraph

In [1]:
from langgraph.graph import StateGraph
from typing import TypedDict, Dict, Any
from IPython.display import display, Image
from langchain_core.runnables.graph import MermaidDrawMethod
import nest_asyncio

class onlineformstate(TypedDict):
    name: str
    mail_id: str
    number: str
    alt_number: str
    submit: str
    have_number: str

def get_name(state):
    state['name'] = input('Enter your name: ')
    return state

def get_mailid(state):
    state['mail_id'] = input('Enter mail id: ')
    return state

def get_number(state):
    state['number'] = input('Enter your phone number: ')
    return state

def validation(state):
    return state

def checklogic1(state):
    num = state['number']
    if len(num) == 10 and num.isdigit() and num[0] in ['6','7','8','9']:
        return 'validated'
    else:
        print('''\nYour number should be digits containing exactly 10 digits and should start either with 6/7/8/9. 
        Please re-enter a valid number''')
        return 'not validated'
        
def router(state):
    return state

def checklogic2(state):
    state['have_number'] = input('Enter yes if you have alternate number')
    if state['have_number'] == 'yes':
        return 'yes' 
    else:
        return 'no'
        
def get_alternate_number(state):
    state['alt_number'] = input('Enter your alternate phone number: ')
    return state

def submit(state):
    state['submit'] = 'Done'
    return state

In [2]:
builder = StateGraph(onlineformstate)
builder.add_node('Enter name',get_name)
builder.add_node('Enter mail id',get_mailid)
builder.add_node('Enter phone number',get_number)
builder.add_node('Validation',validation)
builder.add_node('Have alternate number?',router)
builder.add_node('Enter alternate number',get_alternate_number)
builder.add_node('Submit',submit)

builder.set_entry_point('Enter name')

builder.add_edge('Enter name','Enter mail id')
builder.add_edge('Enter mail id','Enter phone number')
builder.add_edge('Enter phone number','Validation')
builder.add_conditional_edges('Validation',checklogic1,{
    'validated':'Have alternate number?',
    'not validated':'Enter phone number'
})
builder.add_conditional_edges('Have alternate number?',checklogic2,{
    'yes':'Enter alternate number',
    'no':'Submit'
})
builder.add_edge('Enter alternate number','Submit')

builder.set_finish_point('Submit')

graph = builder.compile()

#nest_asyncio.apply()
#display(Image(graph.get_graph().draw_mermaid_png(draw_method=MermaidDrawMethod.PYPPETEER)))

try:
    display(Image(graph.get_graph().draw_mermaid_png(max_retries=5, retry_delay=2.0)))
except Exception as e:
     print("Graph visualization failed:", e)

Graph visualization failed: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 404.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


In [3]:
result = graph.invoke({})
print(result)

Enter your name:  Niv
Enter mail id:  niv97@yahoo.com
Enter your phone number:  0123456789



Your number should be digits containing exactly 10 digits and should start either with 6/7/8/9. 
        Please re-enter a valid number


Enter your phone number:  678901234



Your number should be digits containing exactly 10 digits and should start either with 6/7/8/9. 
        Please re-enter a valid number


Enter your phone number:  6789012345
Enter yes if you have alternate number yes
Enter your alternate phone number:  0123456789


{'name': 'Niv', 'mail_id': 'niv97@yahoo.com', 'number': '6789012345', 'alt_number': '0123456789', 'submit': 'Done'}


In [4]:
result = graph.invoke({})
print(result)

Enter your name:  Niv
Enter mail id:  niv97@yahoo.com
Enter your phone number:  678901234



Your number should be digits containing exactly 10 digits and should start either with 6/7/8/9. 
        Please re-enter a valid number


Enter your phone number:  6789012345
Enter yes if you have alternate number no


{'name': 'Niv', 'mail_id': 'niv97@yahoo.com', 'number': '6789012345', 'submit': 'Done'}
